# Демо: память Python и сборщик мусора

Прокликай Shift+Enter каждую ячейку и посмотри, как Python работает с памятью: ссылочная модель, счётчик ссылок, момент удаления объекта, циклические ссылки, модуль `gc`, ловушка с замыканиями. В конце — три мини-задания.

## Часть 1. `id()` — где живёт объект

У каждого объекта в памяти есть свой адрес. Функция `id(x)` возвращает его как целое число. Если у двух переменных одинаковый `id`, это **тот же объект**, не копия.

In [1]:
x = [1, 2, 3]
y = x                    # ВТОРАЯ ссылка на тот же список
z = [1, 2, 3]            # НОВЫЙ список с теми же значениями

print("id(x):", id(x))
print("id(y):", id(y))   # совпадает с id(x) — это тот же объект
print("id(z):", id(z))   # отличается — это другой объект

print("x is y:", x is y)   # True — один и тот же объект
print("x is z:", x is z)   # False — разные объекты
print("x == z:", x == z)   # True — равные по содержимому


id(x): 4432437120
id(y): 4432437120
id(z): 4432437568
x is y: True
x is z: False
x == z: True


Изменение через одну ссылку видно через другую (потому что объект — общий):

In [2]:
y.append(99)             # меняем через y
print("x:", x)            # x тоже изменился — это один список
print("z:", z)            # z не задет — это другой объект


x: [1, 2, 3, 99]
z: [1, 2, 3]


## Часть 2. Счётчик ссылок и `sys.getrefcount`

У каждого объекта в Python есть **счётчик ссылок** — сколько имён и атрибутов на него указывают. Когда счётчик падает до 0, объект немедленно удаляется. `sys.getrefcount(x)` показывает текущее значение, +1 за временную ссылку самого вызова.

In [3]:
import sys

data = ["a", "b", "c"]   # одна ссылка: data
print("после создания:    ", sys.getrefcount(data))   # обычно 2 (data + аргумент функции)

alias = data                  # вторая ссылка
print("после alias = data:", sys.getrefcount(data))   # +1

container = [data, data]      # ещё две ссылки внутри списка
print("после двойной упаковки:", sys.getrefcount(data))  # +2

del alias
print("после del alias:    ", sys.getrefcount(data))  # -1

del container
print("после del container:", sys.getrefcount(data))  # -2


после создания:     2
после alias = data: 3
после двойной упаковки: 5
после del alias:     4
после del container: 2


Когда последняя ссылка исчезает — объект удаляется. Покажем это через `__del__` (метод, который Python зовёт перед удалением):

In [4]:
class Trace:
    def __init__(self, name):
        self.name = name
        print(f"  создан Trace({self.name!r})")

    def __del__(self):
        print(f"  удалён Trace({self.name!r})")

obj = Trace("X")           # появилось сообщение «создан»
print("работаем с obj...")
obj = None                   # последняя ссылка исчезла → счётчик 0 → удаление
print("obj переприсвоен в None")


  создан Trace('X')
работаем с obj...
  удалён Trace('X')
obj переприсвоен в None


## Часть 3. `del` не всегда удаляет объект

`del name` удаляет **имя**, а не сам объект. Если на объект остались другие ссылки — он живёт. Удаление происходит только когда счётчик достигнет 0.

In [5]:
a = Trace("shared")     # счётчик ссылок: 1
b = a                     # счётчик: 2

print("перед del a:")
del a                     # удаляем имя a; объект жив, b его держит
print("  объект ещё жив (b на него ссылается)")

print("перед del b:")
del b                     # последняя ссылка ушла — объект удаляется
print("  здесь уже должно было выйти 'удалён Trace(...)'")


  создан Trace('shared')
перед del a:
  объект ещё жив (b на него ссылается)
перед del b:
  удалён Trace('shared')
  здесь уже должно было выйти 'удалён Trace(...)'


## Часть 4. Циклические ссылки

А если два объекта ссылаются друг на друга? Счётчики у обоих — `1` (каждый держит другой), но снаружи на них уже никто не указывает. Reference counting сам не справится — объекты остаются в памяти. Это циклическая ссылка.

In [6]:
class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None

    def __del__(self):
        print(f"  удалён Node({self.name!r})")

alpha = Node("alpha")
beta = Node("beta")

alpha.partner = beta       # alpha держит beta
beta.partner = alpha       # beta держит alpha — цикл замкнулся

print("удаляем локальные ссылки...")
del alpha
del beta
print("  ...прошло; __del__ ещё НЕ вызывался")
print("  (объекты живы, потому что держат друг друга)")


удаляем локальные ссылки...
  ...прошло; __del__ ещё НЕ вызывался
  (объекты живы, потому что держат друг друга)


Освободить память помогает **сборщик мусора** — модуль `gc`. Метод `gc.collect()` явно запускает проход и находит циклы:

In [7]:
import gc

print("вызываем gc.collect()...")
collected = gc.collect()
print(f"  ...gc собрал {collected} объектов")


вызываем gc.collect()...
  удалён Node('alpha')
  удалён Node('beta')
  ...gc собрал 4 объектов


## Часть 5. Поколения и пороги в `gc`

Сборщик мусора работает по принципу **поколений**: новые объекты попадают в поколение 0, выжившие — в 1, ещё дольше живущие — в 2. Пороги — сколько новых объектов должно появиться, прежде чем `gc` проверит соответствующее поколение.

In [8]:
import gc

print("пороги:", gc.get_threshold())
# (700, 10, 10) по умолчанию: 700 новых → check gen 0, потом 10 раз gen 0 → check gen 1, etc.

print("количество объектов в каждом поколении:")
for i, n in enumerate(gc.get_count()):
    print(f"  поколение {i}: {n}")


пороги: (700, 10, 10)
количество объектов в каждом поколении:
  поколение 0: 177
  поколение 1: 0
  поколение 2: 0


Запускать `gc.collect()` руками в обычном коде не нужно — Python делает это сам по порогам. Ручной вызов нужен в двух случаях: (а) ML/DL — освободить GPU-память сразу после `del tensor`; (б) измерения производительности — выкинуть мусор перед бенчмарком, чтобы он не помешал.

In [9]:
# Пример: явная очистка перед бенчмарком
import time

gc.collect()                                  # выкидываем накопившийся мусор
start = time.perf_counter()
result = sum(i * i for i in range(1_000_000))  # сама работа
elapsed = time.perf_counter() - start
print(f"работа заняла {elapsed:.3f}s")


работа заняла 0.029s


## Часть 6. Подводный камень: замыкание держит большой объект

Если функция захватывает большой объект из внешней области, этот объект **жив, пока жива функция**. Кажется, мы давно его «отпустили» — а память не освобождается, потому что ссылка живёт внутри функции.

In [10]:
import sys

def make_logger():
    big_table = list(range(1_000_000))   # 1М чисел — несколько мегабайт

    def log(msg):
        # функция использует big_table → захватывает его в замыкание
        return f"[size={len(big_table)}] {msg}"

    return log

logger = make_logger()              # big_table «прикреплён» к logger
print(logger("привет"))             # [size=1000000] привет
print("big_table жив, пока жив logger")

# Освобождаем — теперь big_table исчезнет
logger = None
import gc
gc.collect()
print("logger обнулён → big_table освобождён")


[size=1000000] привет
big_table жив, пока жив logger
logger обнулён → big_table освобождён


Та же ловушка часто встречается в декораторах: декоратор захватывает оригинальную функцию + её аргументы, и пока декорированная функция живёт где-то в коде, аргументы тоже живут.

В ML-коде это критично для тензоров: callback, привязанный к большой модели, держит модель в памяти даже после того, как «обучение закончилось».

## Мини-задания

Три коротких упражнения. Подсказок к именам и API нет — вспомни сам.

**Задание 1.** Создай список `data = [1, 2, 3]` и переменную `alias = data`. Используй встроенный модуль `sys`, чтобы напечатать счётчик ссылок до и после `del alias`. Разница должна быть `1`.

**Задание 2.** Создай два словаря `a` и `b` так, чтобы `a["link"] = b` и `b["link"] = a` (циклическая ссылка). Удали локальные имена и попроси сборщик мусора собрать. Сколько объектов он освободил? Запусти `gc.collect()` и посмотри возвращаемое значение.

**Задание 3.** Что напечатает код ниже? Сначала угадай (что произойдёт со счётчиком ссылок), потом запусти.

In [11]:
# Задание 1
# import sys
# data = [1, 2, 3]
# print("...", sys.getrefcount(data))
# alias = data
# print("...", sys.getrefcount(data))
# del alias
# print("...", sys.getrefcount(data))


In [12]:
# Задание 2
# import gc
# a = {}
# b = {}
# # связать в цикл
# # удалить локальные имена
# # вызвать gc.collect() и напечатать сколько собрал


In [13]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий:
import sys

def make_pair():
    inner = [10, 20, 30]
    return inner, inner          # возвращаем ДВЕ ссылки

x, y = make_pair()
print(sys.getrefcount(x))       # ?
print(x is y)                   # ?
del y
print(sys.getrefcount(x))       # ?


3
True
2
